# Step 3 — Climate Zone Mapping (2024–25)

This notebook assigns an **ASHRAE climate zone** (e.g. `4A`, `5A`, `6B`) to every **operational** school from Step 1.
The climate zone, together with the school level, selects the energy benchmark in a later step.

**Inputs**
- `data/interim/schools_2024_25.csv` (from Step 1).
- A county climate-zone table (see section 1).

**Output:** `data/interim/school_climate_zone_2024_25.csv` — one row per operational school.

## How the mapping works

ASHRAE assigns **one climate zone per county**. Each school is matched to its county by **where it physically sits**:
the state and county come from the school's coordinates (Step 1 columns `location_state_fips` and `county_name`),
not from the jurisdiction that reports it. Bureau of Indian Education schools are therefore matched in the state they
sit in.

County names are written differently in different sources ("Acadia Parish" vs "Acadia", "St. Louis" vs "Saint
Louis", "DeKalb" vs "De Kalb"), so both sides are **normalized** before matching:
lower-case; drop the endings *County, Parish, Borough, Census Area, City and Borough, Municipality, Planning Region*;
write *Saint* as *St*; remove spaces and punctuation. The ending *city* is kept, because Virginia, Maryland and Missouri
have counties and independent cities with the same name (e.g. *Fairfax* and *Fairfax city*, *Baltimore* and
*Baltimore city*).

## 1. The county climate-zone table

This table is **not provided with the project** — extract it yourself from
**ANSI/ASHRAE Standard 169-2020, Addendum a (2021), Table A-4 "Climate Zones for United States Counties"**
(it covers the 50 states and the District of Columbia; the standard is copyrighted and must be obtained from ASHRAE).

Save your extracted table as a CSV at `data/raw/climate_zones/county_climate_zones.csv` with one row per county and
three columns:

| Column | Example | Notes |
|---|---|---|
| `State` | `AL` | two-letter postal code |
| `County` | `Autauga` | county name as written in the table |
| `ClimateZone` | `3A` | zone number 1–8, plus moisture letter A/B/C where the table gives one |

If the column headers in your file are different, change `TABLE_COLUMNS` below.

## 2. Setup

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", context="notebook")


def check(condition: bool, failure_message: str) -> None:
    # Stops the notebook with a plain-language message instead of continuing with bad data.
    if not condition:
        raise ValueError(failure_message)


def climate_zone_sort_key(zone: str) -> tuple[int, str]:
    # Sorts zones in ASHRAE order: 1A, 2A, 2B, 3A, … 8 (number first, then moisture letter).
    return int(zone[0]), zone[1:]

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCHOOLS_FILE = PROJECT_ROOT / "data" / "interim" / "schools_2024_25.csv"
CLIMATE_TABLE_FILE = PROJECT_ROOT / "data" / "raw" / "climate_zones" / "county_climate_zones.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

# Column names in the climate-zone CSV → names used in this notebook.
TABLE_COLUMNS = {"State": "state", "County": "county", "ClimateZone": "climate_zone"}

# Counties that were renamed or split after the climate table was drawn up: name in the school data → name in the
# table. Alaska split Valdez-Cordova Census Area into Chugach and Copper River Census Areas in 2019.
# Applied only when the table still uses the old name.
COUNTY_RENAMES = {
    ("AK", "Chugach Census Area"): "Valdez-Cordova",
    ("AK", "Copper River Census Area"): "Valdez-Cordova",
}

check(
    CLIMATE_TABLE_FILE.exists(),
    f"Climate-zone table not found at {CLIMATE_TABLE_FILE.relative_to(PROJECT_ROOT)}. Save it there as described in section 1.",
)

## 3. Load and check the inputs

In [ ]:
climate_table = pd.read_csv(CLIMATE_TABLE_FILE, dtype=str, keep_default_na=False, na_values=[""])
missing_columns = set(TABLE_COLUMNS) - set(climate_table.columns)
check(
    not missing_columns,
    f"Climate-zone table is missing columns {missing_columns}; its columns are {list(climate_table.columns)}. Update TABLE_COLUMNS.",
)
climate_table = climate_table.rename(columns=TABLE_COLUMNS)[list(TABLE_COLUMNS.values())]
for column in climate_table.columns:
    climate_table[column] = climate_table[column].str.strip()

check(climate_table.notna().all().all(), "Climate-zone table has blank cells.")
invalid_zones = climate_table.loc[
    ~climate_table["climate_zone"].str.fullmatch(r"[1-8][ABC]?"), "climate_zone"
].unique()
check(
    len(invalid_zones) == 0,
    f"Climate-zone table has values that are not ASHRAE zones (1–8 with optional A/B/C): {list(invalid_zones)}.",
)

print(f"Climate table: {len(climate_table):,} counties in {climate_table['state'].nunique()} states/DC")
climate_table["climate_zone"].value_counts().sort_index().to_frame("counties").T

In [ ]:
schools = pd.read_csv(SCHOOLS_FILE, dtype=str, keep_default_na=False, na_values=[""])
schools = schools[schools["is_operational"] == "True"].reset_index(drop=True)

# State FIPS → postal code, read from the data (BIE excluded: "BI" is not a place).
postal_by_fips = (
    schools.loc[schools["admin_state"] != "BI"]
    .drop_duplicates("admin_state_fips")
    .set_index("admin_state_fips")["admin_state"]
)
schools["location_state"] = schools["location_state_fips"].map(postal_by_fips)
check(
    schools["location_state"].notna().all(),
    "Some schools sit in a state FIPS that no jurisdiction reports; update the postal lookup.",
)
check(
    schools["county_name"].notna().all(),
    "Some schools have no county name; Step 1 should have given every school a county.",
)
print(f"Operational schools: {len(schools):,}")

## 4. Match each school to its county's climate zone

In [ ]:
COUNTY_ENDINGS = r"\s+(county|parish|borough|census area|city and borough|municipality|planning region)$"


def normalized_county(name: str) -> str:
    name = re.sub(COUNTY_ENDINGS, "", name.lower().strip())
    name = re.sub(r"\bsaint\b", "st", name)
    return re.sub(r"[^a-z0-9]", "", name)


climate_table["match_key"] = climate_table["state"] + "|" + climate_table["county"].map(normalized_county)
duplicate_keys = climate_table[climate_table["match_key"].duplicated(keep=False)]
check(
    duplicate_keys.empty,
    f"Two counties in the climate table become the same name after normalizing:\n{duplicate_keys}",
)
zone_by_key = climate_table.set_index("match_key")["climate_zone"]

# Use the old name for counties that were renamed or split after the climate table was drawn up.
schools["county_for_matching"] = [
    COUNTY_RENAMES.get((state, county), county)
    for state, county in zip(schools["location_state"], schools["county_name"])
]
schools["match_key"] = schools["location_state"] + "|" + schools["county_for_matching"].map(normalized_county)
schools["climate_zone"] = schools["match_key"].map(zone_by_key)

matched = schools["climate_zone"].notna()
renamed = schools["county_for_matching"] != schools["county_name"]
schools["climate_zone_method"] = None
schools.loc[matched & ~renamed, "climate_zone_method"] = "county_name_match"
schools.loc[matched & renamed, "climate_zone_method"] = "county_renamed"

# Where every county of a state has the same zone, an unmatched county in that state can only be that zone.
# This covers new county names in such states — e.g. Connecticut's 2022 planning regions, when all its old counties share one zone.
zones_per_state = climate_table.groupby("state")["climate_zone"].agg(["nunique", "first"])
single_zone_by_state = zones_per_state.loc[zones_per_state["nunique"] == 1, "first"]
use_state_zone = schools["climate_zone"].isna() & schools["location_state"].isin(single_zone_by_state.index)
schools.loc[use_state_zone, "climate_zone"] = schools.loc[use_state_zone, "location_state"].map(
    single_zone_by_state
)
schools.loc[use_state_zone, "climate_zone_method"] = "state_has_single_zone"

state_in_table = schools["location_state"].isin(climate_table["state"])
schools.loc[schools["climate_zone"].isna() & ~state_in_table, "climate_zone_method"] = "state_not_in_table"
schools.loc[schools["climate_zone"].isna() & state_in_table, "climate_zone_method"] = "county_not_found"

method_order = [
    "county_name_match",
    "county_renamed",
    "state_has_single_zone",
    "county_not_found",
    "state_not_in_table",
]
method_summary = (
    schools["climate_zone_method"].value_counts().reindex(method_order, fill_value=0).to_frame("schools")
)
method_summary["share"] = (method_summary["schools"] / len(schools)).map("{:.2%}".format)
method_summary

| `climate_zone_method` | Meaning |
|---|---|
| `county_name_match` | The school's county was found in the table |
| `county_renamed` | The county was renamed or split (see `COUNTY_RENAMES`); matched under its old name |
| `state_has_single_zone` | County not in the table, but every county of that state has the same zone |
| `county_not_found` | County not in the table and the state has more than one zone — no zone assigned |
| `state_not_in_table` | The state or territory is not in the table — no zone assigned |

In [ ]:
# Every school without a zone, grouped by county, so missing or misspelled table entries can be fixed at the source.
unmatched = (
    schools[schools["climate_zone"].isna()]
    .groupby(["climate_zone_method", "location_state", "county_name"])
    .size()
    .rename("schools")
    .reset_index()
    .sort_values(["climate_zone_method", "schools"], ascending=[True, False])
)
print(f"Counties without a zone: {len(unmatched)} ({unmatched['schools'].sum():,} schools)")
unmatched[unmatched["climate_zone_method"] == "county_not_found"]

In [ ]:
unmatched[unmatched["climate_zone_method"] == "state_not_in_table"].groupby("location_state")[
    "schools"
].sum().to_frame("schools without a zone")

## 5. Call-outs: known limits of this mapping

1. **Territories have no zone.** ASHRAE's county table covers the 50 states and DC only. Schools in Puerto Rico, Guam,
   the Northern Mariana Islands, American Samoa and the U.S. Virgin Islands get no zone.
2. **County names change.** NCES uses current county names; the climate table may use older ones (e.g. Connecticut's
   2022 planning regions, Alaska's 2019 split). Known renames are listed in `COUNTY_RENAMES`; anything else shows up in
   the `county_not_found` list above and should be fixed in the table or added to `COUNTY_RENAMES`.
3. **One zone per county.** A county is assigned a single zone even where its climate varies, e.g. between a valley
   and mountains in the same county.
4. **The table edition matters.** Zone boundaries change between ASHRAE editions. Record which edition the table was
   taken from; this notebook assumes Standard 169-2020 (Addendum a), Table A-4.
5. **Spelling in the table.** A misspelled county in the table (e.g. a missing letter) appears as `county_not_found`;
   correct the table rather than the code.

## 6. Visual checks

In [ ]:
zone_counts = schools["climate_zone"].fillna("no zone").value_counts()
zone_order = sorted(zone_counts.index.drop("no zone", errors="ignore"), key=climate_zone_sort_key)
if "no zone" in zone_counts:
    zone_order.append("no zone")

# Drawn with matplotlib directly: seaborn 0.12's barplot cannot take a plain list under pandas 3.
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(zone_order, zone_counts.reindex(zone_order).values, color="#2b6cb0")
for position, value in enumerate(zone_counts.reindex(zone_order).values):
    ax.text(position, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
ax.set_title("Operational schools by ASHRAE climate zone")
ax.set_xlabel("Climate zone")
ax.set_ylabel("Schools")
plt.tight_layout()
plt.show()

In [ ]:
latitudes = pd.to_numeric(schools["latitude"])
longitudes = pd.to_numeric(schools["longitude"])
contiguous = latitudes.between(24, 50) & longitudes.between(-125, -66)

fig, ax = plt.subplots(figsize=(14, 8))
zones_drawn = [zone for zone in zone_order if zone != "no zone"]
palette = dict(zip(zones_drawn, sns.color_palette("Spectral_r", len(zones_drawn))))
for zone in zones_drawn:
    subset = contiguous & schools["climate_zone"].eq(zone)
    ax.scatter(longitudes[subset], latitudes[subset], s=1.5, color=palette[zone], label=zone)
no_zone = contiguous & schools["climate_zone"].isna()
ax.scatter(longitudes[no_zone], latitudes[no_zone], s=12, color="black", marker="x", label="no zone")
ax.set_title("Climate zone of each operational school (contiguous U.S.)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect(1.25)
ax.legend(title="Zone", markerscale=6, ncol=2, loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

## 7. Save output

`school_climate_zone_2024_25.csv` — one row per operational school:

| Column | Meaning |
|---|---|
| `school_id`, `school_name`, `admin_state` | From Step 1 |
| `location_state`, `county_fips`, `county_name` | State and county the school physically sits in (from its coordinates) |
| `climate_zone` | ASHRAE climate zone; blank when none could be assigned |
| `climate_zone_method` | How the zone was assigned (section 4) |

In [ ]:
OUTPUT_COLUMNS = [
    "school_id",
    "school_name",
    "admin_state",
    "location_state",
    "county_fips",
    "county_name",
    "climate_zone",
    "climate_zone_method",
]
output_path = INTERIM_DIR / "school_climate_zone_2024_25.csv"
schools[OUTPUT_COLUMNS].to_csv(output_path, index=False)

reread = pd.read_csv(output_path, dtype=str)
check(
    len(reread) == len(schools) and reread["school_id"].equals(schools["school_id"]),
    "Saved climate-zone file does not match what was computed.",
)
print(f"Saved {len(reread):,} schools to {output_path.relative_to(PROJECT_ROOT)}")
print(
    f"Schools with a climate zone: {reread['climate_zone'].notna().sum():,} ({reread['climate_zone'].notna().mean():.2%})"
)